# Road Damage Detection: Convert Dataset + Train YOLO Models

This notebook does two jobs:

1. Convert RDD2022 from Pascal VOC XML to YOLO format with a `70/20/10` split.
2. Train and compare `YOLOv8`, `YOLO11`, and `YOLOv10` on the converted dataset.

Split used:

```text
70% train
20% validation
10% test
```


In [1]:
import importlib.util
import subprocess
import sys

required_packages = [
    "ultralytics",
    "torch",
    "cv2",
    "pandas",
    "matplotlib",
    "yaml",
]

pip_packages = {
    "cv2": "opencv-python",
    "yaml": "pyyaml",
}

missing = []
for package in required_packages:
    if importlib.util.find_spec(package) is None:
        missing.append(pip_packages.get(package, package))

if missing:
    print("Installing missing packages:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
else:
    print("All required libraries are installed.")


All required libraries are installed.


In [2]:
import sys
from pathlib import Path
import importlib

import cv2
import matplotlib.pyplot as plt
import pandas as pd
import torch
import yaml
from ultralytics import YOLO

print("Python:", sys.executable)
print("Torch version:", torch.__version__)
print("Torch CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("CUDA is not available in this Python environment.")


Python: c:\Users\MOS3AD\miniconda3\envs\subway_rl\python.exe
Torch version: 2.7.1+cu118
Torch CUDA build: 11.8
CUDA available: True
GPU: NVIDIA GeForce RTX 4070


In [3]:
# Dataset conversion settings
SOURCE_PATH = Path(r"D:\CV_INSTANT\Project 2\RDD2022")
OUTPUT_ROOT = Path(r"D:\CV_INSTANT\Project 2\RDD2022")
DATA_YAML = OUTPUT_ROOT / "RDD2022_YOLO" / "data.yaml"

TRAIN_RATIO = 0.70
VAL_RATIO = 0.20
TEST_RATIO = 1 - TRAIN_RATIO - VAL_RATIO
SEED = 42

# If country zip files still exist, delete them after extraction.
# If they were already deleted, converter.py will reuse OUTPUT_ROOT/extracted.
DELETE_SOURCE_ZIPS = True

# False = reuse RDD2022_YOLO if it already exists.
# Set to True only when you intentionally want to delete and rebuild it.
FORCE_REBUILD_DATASET = False

print("Source:", SOURCE_PATH)
print("Output:", OUTPUT_ROOT)
print(f"Split: train={TRAIN_RATIO:.0%}, val={VAL_RATIO:.0%}, test={TEST_RATIO:.0%}")


Source: D:\CV_INSTANT\Project 2\RDD2022
Output: D:\CV_INSTANT\Project 2\RDD2022
Split: train=70%, val=20%, test=10%


In [4]:
import converter
importlib.reload(converter)

if FORCE_REBUILD_DATASET or not converter.yolo_dataset_ready(DATA_YAML.parent):
    country_folders = converter.prepare_country_folders(
        source=SOURCE_PATH,
        output_root=OUTPUT_ROOT,
        delete_source_zips=DELETE_SOURCE_ZIPS,
    )

    converter.convert_rdd2022_to_yolo(
        country_folders=country_folders,
        output_root=OUTPUT_ROOT,
        train_ratio=TRAIN_RATIO,
        val_ratio=VAL_RATIO,
        seed=SEED,
        force=FORCE_REBUILD_DATASET,
    )
else:
    print("Skipping conversion because DATA_YAML already exists:", DATA_YAML)


Skipping conversion because DATA_YAML already exists: D:\CV_INSTANT\Project 2\RDD2022\RDD2022_YOLO\data.yaml


In [5]:
if not DATA_YAML.exists():
    raise FileNotFoundError(f"data.yaml was not created: {DATA_YAML}")

with DATA_YAML.open("r", encoding="utf-8") as file:
    data_config = yaml.safe_load(file)

dataset_root = Path(data_config["path"])
print("Dataset root:", dataset_root)
print("Classes:", data_config["names"])

for split in ["train", "val", "test"]:
    image_dir = dataset_root / data_config[split]
    label_dir = dataset_root / data_config[split].replace("images", "labels")
    image_count = len([p for p in image_dir.glob("*") if p.suffix.lower() in [".jpg", ".jpeg", ".png"]]) if image_dir.exists() else 0
    label_count = len(list(label_dir.glob("*.txt"))) if label_dir.exists() else 0
    print(f"{split}: {image_count} images, {label_count} labels")


Dataset root: D:\CV_INSTANT\Project 2\RDD2022\RDD2022_YOLO
Classes: {0: 'D00_longitudinal_crack', 1: 'D10_transverse_crack', 2: 'D20_alligator_crack', 3: 'D40_pothole'}
train: 26869 images, 26869 labels
val: 7677 images, 7677 labels
test: 932 images, 932 labels


In [6]:
# YOLO training settings
PROJECT_DIR = Path(r"D:\CV_INSTANT\Project 2\runs\road_damage_yolo_compare")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

MODELS = {
    "YOLOv8": "yolov8s.pt",
    "YOLO11": "yolo11s.pt",
    "YOLOv10": "yolov10s.pt",
}

IMG_SIZE = 640
EPOCHS = 100
BATCH_SIZE = 32
PATIENCE = 20

# Resume behavior
RESUME_IF_POSSIBLE = True
SKIP_COMPLETED_RUNS = True

# Windows + Jupyter can crash PyTorch multiprocessing DataLoader workers.
# Keep this at 0 for stable CUDA training inside the notebook.
WORKERS = 0

# Force CUDA. This intentionally stops if PyTorch cannot see your GPU.
CUDA_DEVICE_INDEX = 0
if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is not available to PyTorch. Your PC has an NVIDIA GPU, but this Python environment "
        "does not have CUDA-enabled PyTorch installed. Install CUDA PyTorch, restart the notebook kernel, "
        "then run again."
    )

torch.cuda.set_device(CUDA_DEVICE_INDEX)
torch.backends.cudnn.benchmark = True

# Ultralytics expects GPU IDs like "0". TORCH_DEVICE is for direct PyTorch calls.
DEVICE = str(CUDA_DEVICE_INDEX)
TORCH_DEVICE = torch.device(f"cuda:{CUDA_DEVICE_INDEX}")

print("Training device:", TORCH_DEVICE)
print("GPU:", torch.cuda.get_device_name(CUDA_DEVICE_INDEX))
print("Resume if possible:", RESUME_IF_POSSIBLE)


Training device: cuda:0
GPU: NVIDIA GeForce RTX 4070
Resume if possible: True


## CUDA PyTorch Install Note

If the previous cell says CUDA is not available, install CUDA-enabled PyTorch in the same environment used by VS Code/Jupyter, then restart the kernel.

Example command for your terminal:

```powershell
python -m pip install --upgrade --force-reinstall torch==2.7.1+cu128 torchvision==0.22.1+cu128 torchaudio==2.7.1+cu128 --index-url https://download.pytorch.org/whl/cu128
```


In [7]:
def run_paths(model_label):
    run_dir = PROJECT_DIR / model_label
    weights_dir = run_dir / "weights"
    return {
        "run_dir": run_dir,
        "weights_dir": weights_dir,
        "last": weights_dir / "last.pt",
        "best": weights_dir / "best.pt",
        "results": run_dir / "results.csv",
    }


def completed_epochs(results_csv):
    if not results_csv.exists():
        return 0

    try:
        df = pd.read_csv(results_csv)
    except Exception:
        return 0

    if df.empty or "epoch" not in df.columns:
        return 0

    # Ultralytics usually stores epoch as 1-based in logs or 0-based in csv depending on version.
    max_epoch = int(df["epoch"].max())
    row_count = len(df)
    return max(max_epoch, row_count)


def run_is_completed(model_label):
    paths = run_paths(model_label)
    if not paths["best"].exists():
        return False
    return completed_epochs(paths["results"]) >= EPOCHS


def train_model(model_label, weights):
    paths = run_paths(model_label)
    print(f"\n========== Training {model_label}: {weights} ==========")

    if SKIP_COMPLETED_RUNS and run_is_completed(model_label):
        print(f"{model_label} already reached {EPOCHS} epochs. Skipping training.")
        print(f"Best weights: {paths['best']}")
        return paths["best"]

    if RESUME_IF_POSSIBLE and paths["last"].exists():
        print(f"Resuming {model_label} from checkpoint:")
        print(paths["last"])
        model = YOLO(str(paths["last"]))
        model.to(TORCH_DEVICE)
        model.train(
            resume=True,
            workers=WORKERS,
            device=DEVICE,
        )
    else:
        print(f"Starting new training run for {model_label}")
        model = YOLO(weights)
        model.to(TORCH_DEVICE)
        model.train(
            data=str(DATA_YAML),
            epochs=EPOCHS,
            imgsz=IMG_SIZE,
            batch=BATCH_SIZE,
            patience=PATIENCE,
            workers=WORKERS,
            device=DEVICE,
            project=str(PROJECT_DIR),
            name=model_label,
            seed=SEED,
            exist_ok=True,
        )

    if not paths["best"].exists():
        raise FileNotFoundError(f"Training finished but best.pt was not found: {paths['best']}")

    print(f"Best weights saved at: {paths['best']}")
    return paths["best"]


def validate_model(model_label, weights_path):
    print(f"\n========== Validating {model_label} on test split ==========")

    model = YOLO(str(weights_path))
    model.to(TORCH_DEVICE)
    metrics = model.val(
        data=str(DATA_YAML),
        split="test",
        imgsz=IMG_SIZE,
        batch=BATCH_SIZE,
        workers=WORKERS,
        device=DEVICE,
        project=str(PROJECT_DIR),
        name=f"{model_label}_test_validation",
        exist_ok=True,
    )

    return {
        "model": model_label,
        "weights": str(weights_path),
        "precision": float(metrics.box.mp),
        "recall": float(metrics.box.mr),
        "mAP50": float(metrics.box.map50),
        "mAP50-95": float(metrics.box.map),
    }


In [11]:
trained_weights = {}
comparison_rows = []

for model_label, weights in MODELS.items():
    best_weights = train_model(model_label, weights)
    trained_weights[model_label] = best_weights
    comparison_rows.append(validate_model(model_label, best_weights))

comparison_df = pd.DataFrame(comparison_rows).sort_values("mAP50-95", ascending=False)
comparison_df



========== Training YOLOv8: yolov8s.pt ==========
YOLOv8 already reached 100 epochs. Skipping training.
Best weights: D:\CV_INSTANT\Project 2\runs\road_damage_yolo_compare\YOLOv8\weights\best.pt

========== Validating YOLOv8 on test split ==========
Ultralytics 8.4.33  Python-3.10.19 torch-2.7.1+cu118 CUDA:0 (NVIDIA GeForce RTX 4070, 12282MiB)
Model summary (fused): 73 layers, 11,127,132 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access  (ping: 0.20.0 ms, read: 24.827.0 MB/s, size: 205.8 KB)
val: Scanning D:\CV_INSTANT\Project 2\RDD2022\RDD2022_YOLO\labels\test.cache... 931 images, 348 backgrounds, 1 corrupt: 100% ━━━━━━━━━━━━ 932/932  0.0s
val: D:\CV_INSTANT\Project 2\RDD2022\RDD2022_YOLO\images\test\India_India_009021.jpg: ignoring corrupt image/label: cannot identify image file 'D:\\CV_INSTANT\\Project 2\\RDD2022\\RDD2022_YOLO\\images\\test\\India_India_009021.jpg'
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 

,model,weights,precision,recall,mAP50,mAP50-95
1,YOLO11,D:\CV_INSTANT\Project 2\runs\road_damage_yolo_...,0.649867,0.565477,0.617355,0.317914
0,YOLOv8,D:\CV_INSTANT\Project 2\runs\road_damage_yolo_...,0.636109,0.550529,0.594021,0.307437
2,YOLOv10,D:\CV_INSTANT\Project 2\runs\road_damage_yolo_...,0.604259,0.549422,0.581112,0.294410


In [12]:
comparison_csv = PROJECT_DIR / "model_comparison_results.csv"
comparison_df.to_csv(comparison_csv, index=False)
print("Saved comparison table:", comparison_csv)

comparison_df[["model", "precision", "recall", "mAP50", "mAP50-95"]]


Saved comparison table: D:\CV_INSTANT\Project 2\runs\road_damage_yolo_compare\model_comparison_results.csv


,model,precision,recall,mAP50,mAP50-95
1,YOLO11,0.649867,0.565477,0.617355,0.317914
0,YOLOv8,0.636109,0.550529,0.594021,0.307437
2,YOLOv10,0.604259,0.549422,0.581112,0.294410


In [13]:
ax = comparison_df.set_index("model")[["precision", "recall", "mAP50", "mAP50-95"]].plot(
    kind="bar",
    figsize=(10, 5),
    rot=0,
)
ax.set_title("YOLO Model Comparison on Road Damage Detection")
ax.set_ylim(0, 1)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


<Figure size 1000x500 with 1 Axes>

In [14]:
best_model_name = comparison_df.iloc[0]["model"]
best_model_path = comparison_df.iloc[0]["weights"]

print("Best model:", best_model_name)
print("Best model weights:", best_model_path)

model = YOLO(best_model_path)
model.to(TORCH_DEVICE)

test_images_dir = dataset_root / data_config["test"]
test_images = sorted(
    [p for p in test_images_dir.glob("*") if p.suffix.lower() in [".jpg", ".jpeg", ".png"]]
)

sample_images = test_images[:20]
print("Prediction samples:", len(sample_images))

model.predict(
    source=[str(p) for p in sample_images],
    imgsz=IMG_SIZE,
    conf=0.25,
    device=DEVICE,
    save=True,
    project=str(PROJECT_DIR),
    name="best_model_predictions",
    exist_ok=True,
)

print("Saved predicted images to:", PROJECT_DIR / "best_model_predictions")


Best model: YOLO11
Best model weights: D:\CV_INSTANT\Project 2\runs\road_damage_yolo_compare\YOLO11\weights\best.pt
Prediction samples: 20

0: 640x640 1 D10_transverse_crack, 1374.2ms
1: 640x640 1 D10_transverse_crack, 1374.2ms
2: 640x640 2 D10_transverse_cracks, 1374.2ms
3: 640x640 1 D10_transverse_crack, 1374.2ms
4: 640x640 (no detections), 1374.2ms
5: 640x640 (no detections), 1374.2ms
6: 640x640 (no detections), 1374.2ms
7: 640x640 3 D00_longitudinal_cracks, 1374.2ms
8: 640x640 2 D00_longitudinal_cracks, 1374.2ms
9: 640x640 1 D00_longitudinal_crack, 1374.2ms
10: 640x640 1 D10_transverse_crack, 1374.2ms
11: 640x640 1 D10_transverse_crack, 1374.2ms
12: 640x640 1 D00_longitudinal_crack, 1374.2ms
13: 640x640 1 D00_longitudinal_crack, 1 D10_transverse_crack, 1374.2ms
14: 640x640 (no detections), 1374.2ms
15: 640x640 4 D00_longitudinal_cracks, 1374.2ms
16: 640x640 1 D00_longitudinal_crack, 1 D10_transverse_crack, 1374.2ms
17: 640x640 1 D10_transverse_crack, 1374.2ms
18: 640x640 1 D20_alli

In [15]:
prediction_dir = PROJECT_DIR / "best_model_predictions"
predicted_images = sorted(
    [p for p in prediction_dir.glob("*") if p.suffix.lower() in [".jpg", ".jpeg", ".png"]]
)

if predicted_images:
    image = cv2.imread(str(predicted_images[0]))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(10, 7))
    plt.imshow(image)
    plt.axis("off")
    plt.title(predicted_images[0].name)
    plt.show()
else:
    print("No prediction images found yet.")


<Figure size 1000x700 with 1 Axes>